In [ ]:
import chromadb
from chromadb.config import Settings
import json

class MyVectorDBConnector:
    def __init__(self, collection_name):
        chroma_client = chromadb.Client(Settings(allow_reset=True))
        self.collection = chroma_client.get_or_create_collection(name=collection_name)

In [1]:
import os
from typing import List
from dotenv import load_dotenv
from openai import OpenAI

class MyVectorDBOpt:
    """向量数据库操作类（基于OpenAI Embeddings API）"""

    def __init__(self, api_key: str, base_url: str, default_model: str = "text-embedding-ada-002"):
        """
        初始化客户端
        :param api_key: API密钥
        :param base_url: API基础URL
        :param default_model: 默认使用的嵌入模型名称
        """
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.default_model = default_model

    def get_embeddings(self, texts: List[str], model: str = None) -> List[List[float]]:
        """
        获取文本的嵌入向量
        :param texts: 文本列表
        :param model: 嵌入模型名称，若不指定则使用默认模型
        :return: 嵌入向量列表
        """
        if not texts:
            return []
        model = model or self.default_model
        try:
            response = self.client.embeddings.create(input=texts, model=model)
            return [item.embedding for item in response.data]
        except Exception as e:
            raise RuntimeError(f"获取嵌入向量失败: {e}")

    def add_embeddings(self, instructions:list[str],embeddings: List[List[float]], collection: MyVectorDBConnector) -> None:
        """
        将嵌入向量添加到向量数据库（待实现具体存储逻辑）
        :param embeddings: 嵌入向量列表
        """
        collection.collection.add(
            documents=instructions,
            embeddings=embeddings,
            ids=[f'id_{i}' for i in range(len(instructions))],
        )

    def search(self, collection: MyVectorDBConnector, query_vector: list[list[float]], top_k: int = 5) -> List[dict]:
        """
        查找数据
        :param collection:
        :param query_vector:
        :param top_k:
        :return:
        """
        result = collection.collection.query(
            query_vector=query_vector,
            top_k = top_k
        )
        raise result


# 使用示例（放在 if __name__ 中避免模块导入时执行）
if __name__ == "__main__":
    load_dotenv()  # 加载  文件中的环境变量
    collection = MyVectorDBConnector("test") # 创建 collection

    # 创建实例（注意保存变量，以便后续调用方法）
    vector_db = MyVectorDBOpt(
        api_key=os.getenv("BASE_EMBEDDINGS_API_KEY", ""),
        base_url=os.getenv("BASE_EMBEDDINGS_API_URL", ""),
        default_model="text-embedding-ada-002"  # 可在此修改默认模型
    )

    # 示例：获取嵌入向量
    sample_texts = ["Hello world", "How to optimize Python code"]
    try:
        # 将我们准备好的数据保存到向量数据库
        embeddings = vector_db.get_embeddings(sample_texts)
        vector_db.add_embeddings(sample_texts,embeddings, collection)

        query = '你好'
        query_vector = vector_db.get_embeddings(query)
        res = vector_db.search(collection, query_vector)
        print(res)
    except Exception as e:
        print(e)

NameError: name 'MyVectorDBConnector' is not defined